# Commercial Building Energy Forecast
End-to-end regression workflow for predicting `energy_consumed_kwh` from commercial building readings.

**The two models compared are Linear Regression and Random Forest Regressor.** The supplied training and test files are used as provided; no additional train/test split is performed.

## 1. Input files and project structure
Place `building_train.csv`, `building_test.csv`, and `building_predict.csv` inside a `data` folder beside this notebook. The notebook creates the output folders automatically.

```text
project/
├── data/
│   ├── building_train.csv
│   ├── building_test.csv
│   └── building_predict.csv
└── solution.ipynb
```

In [ ]:
%pip install pandas numpy scikit-learn joblib

## 2. Import libraries

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)

## 3. Create output folders

In [2]:
for folder in ["processed_data", "artifacts", "output"]:
    os.makedirs(folder, exist_ok=True)

## 4. Data cleaning and feature engineering

The same cleaning and feature-generation logic is applied to all three files.

**Leakage-safe handling of missing values:** this function does not calculate medians separately for each file. Missing numeric values remain as `NaN`; the imputer in the next section learns medians from training data only and reuses them for test and prediction data.

In [ ]:
def prepare_data(input_path, output_path, training=False):
    # Read data and keep the first occurrence of each reading ID.
    df = pd.read_csv(input_path)
    df = df.drop_duplicates(subset="reading_id", keep="first").copy()

    # Parse mixed timestamp formats; invalid timestamps become NaT.
    df["reading_ts"] = pd.to_datetime(
        df["reading_ts"], errors="coerce", format="mixed"
    )
    
    df = df.dropna(subset=["reading_ts"]).copy()

    # Standardize category text.
    categorical_columns = ["building_code", "hvac_mode", "day_type"]
    for column in categorical_columns:
        df[column] = df[column].astype("string").str.strip().str.lower()

    # Convert numeric features; invalid values become NaN.
    numeric_columns = [
        "floor_area_sqm", "occupancy_count", "operating_hours",
        "outdoor_temp_c", "humidity_pct"
    ]
    
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Convert the target when it exists. Prediction data has no target.
    if "energy_consumed_kwh" in df.columns:
        df["energy_consumed_kwh"] = pd.to_numeric(
            df["energy_consumed_kwh"], errors="coerce"
        )

    # Feature engineering.
    df["reading_hour"] = df["reading_ts"].dt.hour
    safe_area = df["floor_area_sqm"].replace(0, np.nan)
    df["occupancy_density"] = df["occupancy_count"] / safe_area

    # Timestamp is no longer required after extracting the hour.
    df = df.drop(columns=["reading_ts"])

    # Convert categories to numeric indicator columns.
    df = pd.get_dummies(df, columns=categorical_columns, dtype=int)

    # Save without the DataFrame index.
    df.to_csv(output_path, index=False)
    return df

## 5. Prepare training, test, and prediction datasets

In [ ]:
train_df = prepare_data(
    "data/building_train.csv",
    "processed_data/building_train_cleaned.csv",
    training=True
)

test_df = prepare_data(
    "data/building_test.csv",
    "processed_data/building_test_cleaned.csv"
)

predict_df = prepare_data(
    "data/building_predict.csv",
    "processed_data/building_predict_cleaned.csv"
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Prediction shape:", predict_df.shape)
display(train_df.head())

## 6. Separate training features and target

`reading_id` is only an identifier, so it is not passed to either model. The target is `energy_consumed_kwh`. Rows without a target cannot be used for supervised training and are removed.

In [ ]:
target_column = "energy_consumed_kwh"
id_column = "reading_id"

train_df = train_df.dropna(subset=[target_column]).copy()

X_train = train_df.drop(columns=[id_column, target_column])
y_train = train_df[target_column]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)


## 7. Fit preprocessing on training data only

The imputer learns each feature's median from `X_train`. The scaler then learns the training means and standard deviations. Both fitted steps are reused unchanged for test and prediction data.

In [ ]:
num_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder())
])

preprocessor = Pipeline([
        (num,num_preprocessor,numeric_columns),(cat,cat_preprocessor,categorical_columns)
])



X_train_processed = preprocessor.fit_transform(X_train)

preprocessor_path = "artifacts/preprocessor.pkl"
joblib.dump(preprocessor, preprocessor_path)

print("Processed training shape:", X_train_processed.shape)

## 8. Train the two specified regression models

In [ ]:
# Model 1: Linear Regression baseline.
linear_model = LinearRegression()
linear_model.fit(X_train_processed, y_train)

# Model 2: Random Forest with the required settings.
forest_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
forest_model.fit(X_train_processed, y_train)

linear_model_path = "artifacts/linear_regression.pkl"
forest_model_path = "artifacts/random_forest.pkl"

joblib.dump(linear_model, linear_model_path)
joblib.dump(forest_model, forest_model_path)
=;0o.

print("Both models trained and saved.")

## 9. Prepare the held-back test data

The one-hot encoded test columns may differ from the training columns. `reindex()` restores the training column names and order, inserts zero for missing dummy columns, and drops unexpected columns.

Crucially, use `transform()` here, not `fit_transform()`. The test set must not determine imputation or scaling parameters.

In [ ]:
test_df = test_df.dropna(subset=[target_column]).copy()

X_test = test_df.drop(columns=[id_column, target_column])
y_test = test_df[target_column]

X_test = X_test.reindex(
    columns=preprocessor.feature_names_in_,
    fill_value=0
)

X_test_processed = preprocessor.transform(X_test)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

## 10. Evaluate both models

- **MAE:** average absolute prediction error, in kWh.
- **RMSE:** square root of mean squared error, in kWh; larger errors have a stronger effect.
- **R²:** proportion of target variation explained by the model.

The model with the lowest test RMSE is selected.

In [ ]:
linear_predictions = linear_model.predict(X_test_processed)
forest_predictions = forest_model.predict(X_test_processed)

results = []

for model_name, predictions in [
    ("linear_regression", linear_predictions),
    ("random_forest", forest_predictions)
]:
    results.append({
        "model_name": model_name,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions)
    })

comparison = pd.DataFrame(results).sort_values(
    by="rmse", ascending=True
).reset_index(drop=True)

comparison.to_csv("output/model_comparison.csv", index=False)
display(comparison)

## 11. Select the model with the lowest RMSE

In [ ]:
best_model_name = comparison.loc[0, "model_name"]

if best_model_name == "random_forest":
    selected_model_path = forest_model_path
else:
    selected_model_path = linear_model_path

print("Selected model:", selected_model_path)

## 12. Predict electricity consumption for new readings

The prediction file has no target column. Its features are aligned to the training columns, transformed using the fitted training preprocessor, and passed to the selected model. The input reading order is preserved.

In [ ]:
prediction_ids = predict_df[id_column].copy()

X_new = predict_df.drop(
    columns=[id_column, target_column],
    errors="ignore"
)

X_new = X_new.reindex(
    columns=preprocessor.feature_names_in_,
    fill_value=0
)

X_new_processed = preprocessor.transform(X_new)

selected_model = joblib.load(selected_model_path)
new_predictions = selected_model.predict(X_new_processed)

# Electricity consumption cannot be negative.
new_predictions = np.round(np.maximum(new_predictions, 0), 1)

prediction_results = pd.DataFrame({
    "reading_id": prediction_ids,
    "predicted_energy_consumed_kwh": new_predictions
})

prediction_results.to_csv(
    "output/building_predictions.csv",
    index=False
)

display(prediction_results.head())
print("Predictions written:", prediction_results.shape)

## 13. Generated files and final checks

After successful execution:

```text
processed_data/
├── building_train_cleaned.csv
├── building_test_cleaned.csv
└── building_predict_cleaned.csv

artifacts/
├── preprocessor.pkl
├── linear_regression.pkl
└── random_forest.pkl

output/
├── model_comparison.csv
└── building_predictions.csv
```

Check that the comparison contains two models, the selected model has the lowest RMSE, and the prediction output has the required two columns. The number and order of prediction rows should match the valid input prediction records.

Metric values are calculated from the supplied data; none are hard-coded.